In [34]:
import os, re, json
# để làm pali-vi, đầu tiên lấy file của pali-vi, ra kết quả (json),
# maping giữa pali-vi và tmc và json của c-pali-tmc-vi (nhớ đổi mn-> mnc)


# Thư mục chứa các file .md",",
SOURCE_DIR = "../../docs/kinhtruongbo/thichminhchau"

# Danh sách tên file .md",", cần xử lý (chỉ tên file, không cần đường dẫn đầy đủ)
FILES = [
"dn-001-kinh-pham-vong.md",
"dn-002-kinh-sa-mon-qua.md",
"dn-003-kinh-ambattha-a-ma-tru.md",
"dn-004-kinh-sonadanda-chung-duc.md",
"dn-005-kinh-kutadanta-cuu-la-dan-dau.md",
"dn-006-kinh-mahali.md",
"dn-007-kinh-jaliya.md",
"dn-008-kinh-ca-diep-su-tu-hong.md",
"dn-009-kinh-potthapada-bo-sa-ba-lau.md",
"dn-010-kinh-subha-tu-ba.md",
"dn-011-kinh-kevaddha-kien-co.md",
"dn-012-kinh-lohicca-lo-hi-gia.md",
"dn-013-kinh-tevijja-tam-minh.md",
"dn-014-kinh-dai-bon.md",
"dn-015-kinh-dai-duyen.md",
"dn-016-kinh-dai-bat-niet-ban.md",
"dn-017-kinh-dai-thien-kien-vuong.md",
"dn-018-kinh-xa-ni-sa.md",
"dn-019-kinh-dai-dien-ton.md",
"dn-020-kinh-dai-hoi.md",
"dn-021-kinh-de-thich-so-van.md",
"dn-022-kinh-dai-niem-xu.md",
"dn-023-kinh-te-tuc.md",
"dn-024-kinh-ba-le.md",
"dn-025-kinh-uu-dam-ba-la-su-tu-hong.md",
"dn-026-kinh-chuyen-luan-thanh-vuong-su-tu-hong.md",
"dn-027-kinh-khoi-the-nhan-bon.md",
"dn-028-kinh-tu-hoan-hy.md",
"dn-029-kinh-thanh-tinh.md",
"dn-030-kinh-tuong.md",
"dn-031-kinh-giao-tho-thi-ca-la-viet.md",
"dn-032-kinh-a-sa-nang-chi.md",
"dn-033-kinh-phung-tung.md",
"dn-034-kinh-thap-thuong.md",
]

# FILES = [

#     "dn-001-kinh-luoi-troi.md",
# "dn-002-kinh-ve-thanh-qua-cua-nguoi-tu-hanh.md",
# "dn-003-kinh-ambattha.md",
# "dn-004-kinh-sonadanda.md",
# "dn-005-kinh-kutadanta.md",
# "dn-006-kinh-mahali.md",
# "dn-007-kinh-jaliya.md",
# "dn-008-kinh-tieng-gam-su-tu.md",
# "dn-009-kinh-potthapada.md",
# "dn-010-kinh-subha.md",
# "dn-011-kinh-kevatta.md",
# "dn-012-kinh-lohicca.md",
# "dn-013-kinh-tevijja-kinh-ba-minh.md",
# "dn-014-kinh-dai-ve-nguon-goc.md",
# "dn-015-kinh-dai-ve-quan-he-phu-thuoc.md",
# "dn-016-kinh-dai-ve-bat-niet-ban.md",
# "dn-017-kinh-ai-thien-kien-vuong.md",
# "dn-018-kinh-janavasabha.md",
# "dn-019-kinh-dai-ve-govinda.md",
# "dn-020-kinh-ai-hoi.md",
# "dn-021-kinh-sakkapanha-e-thich-so-van.md",
# "dn-022-kinh-dai-ve-bon-noi-chu-tam.md",
# "dn-023-kinh-payasi.md",
# "dn-024-kinh-pathika.md",
# "dn-025-kinh-udumbarika.md",
# "dn-026-kinh-vua-chuyen-luan.md",
# "dn-027-kinh-ve-nguon-goc-toi-so.md",
# "dn-028-kinh-niem-tin-trong-sang.md",
# "dn-029-kinh-ro-rang-va-mach-lac.md",
# "dn-030-kinh-ac-iem.md",
# "dn-031-kinh-singala.md",
# "dn-032-kinh-atanatiya.md",
# "dn-033-kinh-tung-oc-cung-nhau.md",
# "dn-034-kinh-den-muoi.md",
# ]

# Heading cấp mấy được coi là "đoạn con" (### = 3, ## = 2, ...)
CHILD_HEADING_LEVEL = 3


## 2. Các hàm xử lý

In [35]:
import os
import re

CHILD_HEADING_LEVEL = 6  # Tùy chỉnh cấp độ heading con mong muốn
TOP_INDEX_RE = re.compile(r"^[a-z]+-0*(\d+)")
H1_RE = re.compile(r"^#\s+(.*)$", re.MULTILINE)
CHILD_RE = re.compile(r"^#{2,%d}\s+(.*)$" % CHILD_HEADING_LEVEL)

# Regex tìm anchor dạng {#...} ở cuối heading
ANCHOR_RE = re.compile(r"\{#([^}]+)\}\s*$")


def top_index_from_slug(slug):
    m = TOP_INDEX_RE.match(slug.lower())
    return m.group(1) if m else None


def extract_h1(text):
    m = H1_RE.search(text)
    return m.group(1).strip() if m else None


def process_file(dirpath, filename, items, warnings):
    slug = filename[:-3] if filename.endswith(".md") else filename
    filepath = os.path.join(dirpath, filename)
    if not os.path.isfile(filepath):
        warnings.append(f"{filename}: không tìm thấy file, bỏ qua")
        return

    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    top_index = top_index_from_slug(slug)
    title = extract_h1(text)

    item = {}
    item["title"] = title if title else "??? (không tìm thấy H1)"
    if not title:
        warnings.append(f"{filename}: không tìm thấy dòng H1 (# ...)")
    item["slug"] = slug
    item["children"] = []  # Đổi thành mảng chứa danh sách anchor

    for line in text.splitlines():
        m = CHILD_RE.match(line)
        if not m:
            continue
        heading_text = m.group(1).strip()

        # Bắt anchor dạng {#...}
        anchor_match = ANCHOR_RE.search(heading_text)
        if anchor_match:
            anchor = anchor_match.group(1).strip()
            item["children"].append(anchor)

    if top_index is None:
        warnings.append(
            f"{filename}: không suy ra được số thứ tự từ tên file, cần điền tay"
        )
        key = f"???({slug})"
    else:
        key = str(int(top_index))
        if key in items:
            warnings.append(
                f'{filename}: trùng key "{key}" với 1 file khác — kiểm tra lại'
            )

    items[key] = item

## 3. Chạy xử lý

In [36]:
items = {}
warnings = []

for fn in FILES:
    process_file(SOURCE_DIR, fn, items, warnings)

if warnings:
    print("⚠️  Cảnh báo:")
    for w in warnings:
        print(" -", w)
else:
    print("Không có cảnh báo.")


Không có cảnh báo.


In [37]:
output = {
    "folder": "kinhtruongbo",
    "editions": {
        "tmc": {
            "label": "Pali (Vi)",
            "path": "pali-vi",
            "index_length": "2",
            "items": items,
        }
    },
}

print(json.dumps(items, ensure_ascii=False, indent=2))


{
  "1": {
    "title": "1. KINH PHẠM VÕNG",
    "slug": "dn-001-kinh-pham-vong",
    "children": []
  },
  "2": {
    "title": "2. KINH SA MÔN QỦA",
    "slug": "dn-002-kinh-sa-mon-qua",
    "children": []
  },
  "3": {
    "title": "3. KINH AMBATTHA (A-MA-TRÚ)",
    "slug": "dn-003-kinh-ambattha-a-ma-tru",
    "children": []
  },
  "4": {
    "title": "4. KINH SONADANDA (CHỦNG ÐỨC)",
    "slug": "dn-004-kinh-sonadanda-chung-duc",
    "children": []
  },
  "5": {
    "title": "5. KINH KÙTADANTA (CỨU-LA-ÐÀN-ÐẦU)",
    "slug": "dn-005-kinh-kutadanta-cuu-la-dan-dau",
    "children": []
  },
  "6": {
    "title": "6. KINH MAHÀLI",
    "slug": "dn-006-kinh-mahali",
    "children": []
  },
  "7": {
    "title": "7. KINH JÀLIYA",
    "slug": "dn-007-kinh-jaliya",
    "children": []
  },
  "8": {
    "title": "8. KINH CA-DIẾP SƯ TỬ HỐNG",
    "slug": "dn-008-kinh-ca-diep-su-tu-hong",
    "children": []
  },
  "9": {
    "title": "9. KINH POTTHAPÀDA (BỐ-SÁ-BÀ-LÂU)",
    "slug": "dn-009-kinh-po

## 5. (Tùy chọn) Ghi ra file JSON

Chạy cell dưới nếu muốn lưu kết quả ra file thay vì chỉ copy từ output ở trên.

In [ ]:
OUT_PATH = "quicklink-data.generated.json"

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Đã ghi: {OUT_PATH}")
